# XGBoost

### Import needed libraries

In [55]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb


### Initialize dataframes with aggregate data

In [80]:
phy30sdf = pd.read_csv('/Users/tatum/Downloads/testbed_system_1_aggregates/phys_agg_30s.csv', encoding="utf-8-sig")
phy16sdf = pd.read_csv('/Users/tatum/Downloads/testbed_system_1_aggregates/phys_agg_16s.csv', encoding="utf-8-sig")
phy10sdf = pd.read_csv('/Users/tatum/Downloads/testbed_system_1_aggregates/phys_agg_10s.csv', encoding="utf-8-sig")
scada30sdf = pd.read_csv('/Users/tatum/Downloads/testbed_system_1_aggregates/scada_agg_30s.csv', encoding="utf-8-sig")
scada16sdf = pd.read_csv('/Users/tatum/Downloads/testbed_system_1_aggregates/scada_agg_16s.csv', encoding="utf-8-sig")
scada10sdf = pd.read_csv('/Users/tatum/Downloads/testbed_system_1_aggregates/scada_agg_10s.csv', encoding="utf-8-sig")

phy30sdf.head()


,bucket,system_id,prop_key,asset_id,avg_value,max_value,min_value,num_measurements,num_attacks,attack_types
0,2021-04-09 11:30:30+00:00,testbed_system_1,pressure,testbed_system_1_Tank_Tank_1,31.8,178.0,0.0,10,0,['normal']
1,2021-04-09 11:30:30+00:00,testbed_system_1,pressure,testbed_system_1_Tank_Tank_2,0.0,0.0,0.0,10,0,['normal']
2,2021-04-09 11:30:30+00:00,testbed_system_1,pressure,testbed_system_1_Tank_Tank_3,0.0,0.0,0.0,10,0,['normal']
3,2021-04-09 11:30:30+00:00,testbed_system_1,pressure,testbed_system_1_Tank_Tank_4,0.0,0.0,0.0,10,0,['normal']
4,2021-04-09 11:30:30+00:00,testbed_system_1,pressure,testbed_system_1_Tank_Tank_5,0.0,0.0,0.0,10,0,['normal']


### Extract feature and target arrays

In [57]:
X, y = phy30sdf.drop('num_attacks', axis=1), phy30sdf[['num_attacks']]

### Convert text features to categories

In [58]:
phy30sdf = X.select_dtypes(exclude=np.number).columns.tolist()

for col in phy30sdf:
   X[col] = X[col].astype('category')

### Split the data

In [59]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1)

### Convert into DMatrix

In [60]:
dtrain_reg = xgb.DMatrix(X_train, y_train, enable_categorical=True)
dtest_reg = xgb.DMatrix(X_test, y_test, enable_categorical=True)

### Define hyperparameters

In [61]:
# Define hyperparameters
params = {"objective": "reg:squarederror", "tree_method": "hist"}

In [ ]:
n = 1000
model = xgb.train(
   params=params,
   dtrain=dtrain_reg,
   num_boost_round=n,
)

In [63]:
preds = model.predict(dtest_reg)

In [98]:
params = {"objective": "reg:squarederror", "tree_method": "hist"}

results = xgb.cv(
   params, dtrain_reg,
   num_boost_round=n,
   nfold=5,
   early_stopping_rounds=39
)

[0]	validation-rmse:6.98942	train-rmse:7.43601
[250]	validation-rmse:0.00001	train-rmse:0.00001
[500]	validation-rmse:0.00001	train-rmse:0.00001
[750]	validation-rmse:0.00001	train-rmse:0.00001
[1000]	validation-rmse:0.00001	train-rmse:0.00001
[1250]	validation-rmse:0.00001	train-rmse:0.00001
[1500]	validation-rmse:0.00001	train-rmse:0.00001
[1750]	validation-rmse:0.00001	train-rmse:0.00001
[2000]	validation-rmse:0.00001	train-rmse:0.00001
[2250]	validation-rmse:0.00001	train-rmse:0.00001
[2500]	validation-rmse:0.00001	train-rmse:0.00001
[2750]	validation-rmse:0.00001	train-rmse:0.00001
[3000]	validation-rmse:0.00001	train-rmse:0.00001
[3250]	validation-rmse:0.00001	train-rmse:0.00001
[3500]	validation-rmse:0.00001	train-rmse:0.00001
[3750]	validation-rmse:0.00001	train-rmse:0.00001
[4000]	validation-rmse:0.00001	train-rmse:0.00001
[4250]	validation-rmse:0.00001	train-rmse:0.00001
[4500]	validation-rmse:0.00001	train-rmse:0.00001
[4750]	validation-rmse:0.00001	train-rmse:0.00001
[4999]

In [101]:
# quick inspection
print(results[['train-rmse-mean','test-rmse-mean']].head(8))
print('\nSummary:')
print(results[['train-rmse-mean','test-rmse-mean']].describe())
print('\nNa counts:')
print(results[['train-rmse-mean','test-rmse-mean']].isna().sum())

# min/max ranges to see if one is far away or identical
tmin, tmax = results['train-rmse-mean'].min(), results['train-rmse-mean'].max()
vmin, vmax = results['test-rmse-mean'].min(), results['test-rmse-mean'].max()
print(f' train min/max: {tmin:.6g} / {tmax:.6g}')
print(f' test  min/max: {vmin:.6g} / {vmax:.6g}')

   train-rmse-mean  test-rmse-mean
0         7.437103        7.437597
1         5.211462        5.211367
2         3.652124        3.652219
3         2.559126        2.559032
4         1.793288        1.792770
5         1.256638        1.256418
6         0.880613        0.880458
7         0.617103        0.616895

Summary:
       train-rmse-mean  test-rmse-mean
count        40.000000       40.000000
mean          0.621312        0.621281
std           1.546166        1.546206
min           0.000014        0.000014
25%           0.000229        0.000229
50%           0.007357        0.007358
75%           0.235011        0.234929
max           7.437103        7.437597

Na counts:
train-rmse-mean    0
test-rmse-mean     0
dtype: int64
 train min/max: 1.35763e-05 / 7.4371
 test  min/max: 1.35872e-05 / 7.4376
